In [ ]:

# Optional dependency install for a fresh environment
# %pip install -q -U stable-baselines3[extra] gymnasium[box2d] jupyter nbconvert matplotlib pandas scipy


In [1]:
import os
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

warnings.filterwarnings("ignore")

SEEDS = [0, 1, 2, 3, 4]
ENVS_TO_USE = {
    "CartPole-v1": 100_000,
    "LunarLander-v3": 250_000,
    "MountainCar-v0": 200_000,
}
SUCCESS_THRESHOLDS = {
    "CartPole-v1": 475,
    "LunarLander-v3": 200,
    "MountainCar-v0": -110,
}

EVAL_FREQ = 5_000
EVAL_EPISODES = 10

# All result artifacts are written to the current working folder.
BASE_DIR = Path.cwd().resolve()
RESULTS_DIR = BASE_DIR / "results"
ARTIFACT_DIR = RESULTS_DIR / "artifacts"
METRIC_DIR = RESULTS_DIR / "metrics"
FIGURE_DIR = RESULTS_DIR / "figures"
LOG_DIR = RESULTS_DIR / "logs"
MODEL_DIR = BASE_DIR / "models"

for _d in [RESULTS_DIR, ARTIFACT_DIR, METRIC_DIR, FIGURE_DIR, LOG_DIR, MODEL_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


def out_path(filename: str) -> str:
    ext = Path(filename).suffix.lower()
    if ext == ".pkl":
        return str(ARTIFACT_DIR / filename)
    if ext == ".csv":
        return str(METRIC_DIR / filename)
    if ext in {".png", ".jpg", ".jpeg", ".svg", ".pdf"}:
        return str(FIGURE_DIR / filename)
    if ext == ".log":
        return str(LOG_DIR / filename)
    return str(RESULTS_DIR / filename)


print(f"Gymnasium: {gym.__version__}")
print(f"Stable Baselines3: {sb3.__version__}")
print(f"Seeds: {SEEDS}")
print(f"Environments: {list(ENVS_TO_USE.keys())}")
print(f"Working directory: {BASE_DIR}")
print(f"Results folder: {RESULTS_DIR}")
print(f"Model folder: {MODEL_DIR}")


Gymnasium: 1.2.3
Stable Baselines3: 2.7.1
Seeds: [0, 1, 2, 3, 4]
Environments: ['CartPole-v1', 'LunarLander-v3', 'MountainCar-v0']
Working directory: E:\Planning_AI\Project
Artifacts folder: E:\Planning_AI\Project
Model folder: E:\Planning_AI\Project\models


In [2]:
class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self._current_reward = 0.0

    def _on_step(self):
        self._current_reward += float(self.locals["rewards"][0])
        if bool(self.locals["dones"][0]):
            self.episode_rewards.append(self._current_reward)
            self._current_reward = 0.0
        return True


class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_fn):
        super().__init__(env)
        self.reward_fn = reward_fn
        self._last_obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_obs = obs
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        try:
            shaped = float(self.reward_fn(self._last_obs, action, reward, obs, terminated or truncated, info))
            shaped = float(np.clip(shaped, -100.0, 100.0))
        except Exception:
            shaped = float(reward)
        self._last_obs = obs
        return obs, shaped, terminated, truncated, info


class DefaultEvalCallback(BaseCallback):
    def __init__(self, env_id, eval_freq=5_000, n_eval_episodes=10):
        super().__init__()
        self.env_id = env_id
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.eval_steps = []
        self.eval_returns = []

    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            eval_env = gym.make(self.env_id)
            mean_r, _ = evaluate_policy(
                self.model,
                eval_env,
                n_eval_episodes=self.n_eval_episodes,
                deterministic=True,
            )
            eval_env.close()
            self.eval_steps.append(int(self.num_timesteps))
            self.eval_returns.append(float(mean_r))
        return True


def train_agent(env_id, total_timesteps, seed, reward_fn=None, label="agent"):
    def make_train_env():
        env = gym.make(env_id)
        if reward_fn is not None:
            env = CustomRewardWrapper(env, reward_fn)
        return Monitor(env)

    train_env = DummyVecEnv([make_train_env])
    reward_cb = RewardLogger()
    eval_cb = DefaultEvalCallback(env_id, eval_freq=EVAL_FREQ, n_eval_episodes=EVAL_EPISODES)
    callback = CallbackList([reward_cb, eval_cb])

    model = PPO(
        "MlpPolicy",
        train_env,
        verbose=0,
        seed=seed,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
    )

    print(f"[{label}] env={env_id} seed={seed} steps={total_timesteps}")
    model.learn(total_timesteps=total_timesteps, callback=callback)

    model_path = str(MODEL_DIR / f"{env_id.replace('-', '_')}_{label}_s{seed}.zip")
    model.save(model_path)
    train_env.close()

    return {
        "seed": seed,
        "train_rewards": reward_cb.episode_rewards,
        "eval_steps": eval_cb.eval_steps,
        "eval_returns": eval_cb.eval_returns,
        "model_path": model_path,
    }


def run_experiment(env_id, total_timesteps, reward_fn=None, label="agent", seeds=None):
    seeds = SEEDS if seeds is None else seeds
    runs = []
    for seed in seeds:
        run = train_agent(
            env_id=env_id,
            total_timesteps=total_timesteps,
            seed=seed,
            reward_fn=reward_fn,
            label=label,
        )
        runs.append(run)
    return runs


def smooth(values, window=5):
    if len(values) < window:
        return values
    arr = np.array(values, dtype=float)
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode="valid").tolist()


In [3]:
def get_human_reward_function(env_id):
    if env_id == "CartPole-v1":
        code = '''
def custom_reward(obs, action, default_reward, next_obs, done, info):
    angle_penalty = abs(obs[2]) / 0.418
    pos_penalty = abs(obs[0]) / 4.8
    reward = 1.0 - 0.8 * angle_penalty - 0.2 * pos_penalty
    if done:
        reward = -10.0
    return float(reward)
'''
    elif env_id == "LunarLander-v3":
        code = '''
def custom_reward(obs, action, default_reward, next_obs, done, info):
    upright = 1.0 - abs(next_obs[4]) / 3.14159
    centered = 1.0 - abs(next_obs[0])
    descent = 1.0 - abs(next_obs[3])
    leg_bonus = next_obs[6] + next_obs[7]
    reward = default_reward + 0.4 * upright + 0.2 * centered + 0.2 * descent + 0.5 * leg_bonus
    return float(reward)
'''
    elif env_id == "MountainCar-v0":
        code = '''
def custom_reward(obs, action, default_reward, next_obs, done, info):
    progress = (next_obs[0] - obs[0]) * 100.0
    velocity = abs(next_obs[1]) * 10.0
    reward = default_reward + progress + velocity
    if done and next_obs[0] >= 0.5:
        reward += 100.0
    return float(reward)
'''
    else:
        raise ValueError(f"No human reward for {env_id}")

    namespace = {}
    exec(compile(code.strip(), "<human_reward>", "exec"), namespace)
    fn = [obj for name, obj in namespace.items() if callable(obj) and not name.startswith("_")][0]
    return fn, code.strip()


In [6]:

# Local LLM setup (Ollama)
import os
import json as _json
import urllib.request
import urllib.error

USE_LOCAL_OLLAMA = True
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:1.5b")


def ollama_generate(prompt, system_instruction, temperature=0.2, max_output_tokens=1024):
    url = OLLAMA_HOST.rstrip("/") + "/api/generate"
    full_prompt = f"System:\n{system_instruction}\n\nUser:\n{prompt}\n"
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_output_tokens,
        },
    }
    data = _json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=300) as resp:
        body = _json.loads(resp.read().decode("utf-8"))
    return body.get("response", ""), body


# Health check
_test_text, _test_raw = ollama_generate(
    prompt="Reply with exactly OK.",
    system_instruction="You are concise.",
    temperature=0.0,
    max_output_tokens=16,
)
print(f"Local Ollama ready. host={OLLAMA_HOST}, model={OLLAMA_MODEL}, test={_test_text.strip()}")


Note: you may need to restart the kernel to use updated packages.
Gemini client disabled: set GOOGLE_CLOUD_PROJECT for Vertex AI or GEMINI_API_KEY/GOOGLE_API_KEY for Gemini API.


In [7]:

FAILURE_TAXONOMY = {
    "syntax_runtime_invalid": "Syntax error, compile failure, missing function, or runtime test failure",
    "truncation_incomplete": "Output ended before complete valid function",
    "underspecification": "Generated reward misses key task terms/state features",
    "reward_hacking": "Good shaped-training behavior but poor default-eval behavior",
    "magnitude_mismatch": "Reward scale dominates or collapses learning signal",
    "correct": "No obvious failure under current metrics",
}


def get_llm_reward_function(env_id, prompt, model_name=None, max_retries=3):
    model_name = model_name or OLLAMA_MODEL

    system_instruction = (
        "You are an expert RL researcher. Return only valid Python code for one function. "
        "No markdown, no backticks, no imports, no docstrings."
    )

    last_error = None
    code = ""

    for attempt in range(max_retries):
        try:
            code, _raw = ollama_generate(
                prompt=prompt,
                system_instruction=system_instruction,
                temperature=0.2,
                max_output_tokens=1024,
            )
            code = (code or "").replace("```python", "").replace("```", "").strip()
            break
        except Exception as exc:
            last_error = str(exc)
            time.sleep(5 * (attempt + 1))

    if not code:
        return None, code, {
            "status": "syntax_runtime_invalid",
            "detail": f"Local LLM failure: {last_error}",
            "finish_reason": "API_ERROR",
            "model": model_name,
        }

    if code.count("def ") == 0 or ("return" not in code and len(code) > 0):
        return None, code, {
            "status": "truncation_incomplete",
            "detail": "Likely incomplete output",
            "finish_reason": "UNKNOWN",
            "model": model_name,
        }

    if not code.startswith("def "):
        return None, code, {
            "status": "syntax_runtime_invalid",
            "detail": "Output does not start with def",
            "finish_reason": "UNKNOWN",
            "model": model_name,
        }

    namespace = {}
    try:
        exec(compile(code, "<llm_reward>", "exec"), namespace)
    except Exception as exc:
        return None, code, {
            "status": "syntax_runtime_invalid",
            "detail": f"Compile error: {exc}",
            "finish_reason": "UNKNOWN",
            "model": model_name,
        }

    fn = None
    for name, obj in namespace.items():
        if callable(obj) and not name.startswith("_"):
            fn = obj
            break

    if fn is None:
        return None, code, {
            "status": "syntax_runtime_invalid",
            "detail": "No callable reward function found",
            "finish_reason": "UNKNOWN",
            "model": model_name,
        }

    try:
        dummy = np.zeros(8, dtype=float)
        float(fn(dummy, 0, 1.0, dummy, False, {}))
    except Exception as exc:
        return None, code, {
            "status": "syntax_runtime_invalid",
            "detail": f"Runtime test failed: {exc}",
            "finish_reason": "UNKNOWN",
            "model": model_name,
        }

    return fn, code, {
        "status": "ok",
        "detail": "valid function",
        "finish_reason": "STOP",
        "model": model_name,
    }


In [8]:
PROMPTS = {}

PROMPTS["CartPole-v1"] = '''
Write a complete Python reward function for CartPole-v1.
Observation:
- obs[0]: cart position
- obs[1]: cart velocity
- obs[2]: pole angle
- obs[3]: pole angular velocity

Rules:
- Use exactly this signature:
  def custom_reward(obs, action, default_reward, next_obs, done, info):
- Return a float
- No imports
- No docstrings
- Keep it concise
'''

PROMPTS["LunarLander-v3"] = '''
Write a complete Python reward function for LunarLander-v3.
Observation:
- obs[0]: x position
- obs[1]: y position
- obs[2]: x velocity
- obs[3]: y velocity
- obs[4]: angle
- obs[5]: angular velocity
- obs[6]: left leg contact
- obs[7]: right leg contact

Rules:
- Use exactly this signature:
  def custom_reward(obs, action, default_reward, next_obs, done, info):
- Return a float
- No imports
- No docstrings
- Keep it concise
'''

PROMPTS["MountainCar-v0"] = '''
Write a complete Python reward function for MountainCar-v0.
Observation:
- obs[0]: position
- obs[1]: velocity

Rules:
- Use exactly this signature:
  def custom_reward(obs, action, default_reward, next_obs, done, info):
- Return a float
- No imports
- No docstrings
- Keep it concise
'''

print("Prompt coverage:")
for env_id in ENVS_TO_USE:
    print(env_id, "OK" if env_id in PROMPTS else "MISSING")


Prompt coverage:
CartPole-v1 OK
LunarLander-v3 OK
MountainCar-v0 OK


In [9]:

default_path = out_path("default_results.pkl")
human_path = out_path("human_results.pkl")
human_codes_path = out_path("human_codes.pkl")

if os.path.exists(default_path) and os.path.exists(human_path) and os.path.exists(human_codes_path):
    with open(default_path, "rb") as f:
        default_results = pickle.load(f)
    with open(human_path, "rb") as f:
        human_results = pickle.load(f)
    with open(human_codes_path, "rb") as f:
        human_codes = pickle.load(f)
    print("Loaded existing default + human results (skipping retraining).")
else:
    default_results = {}
    human_results = {}
    human_codes = {}

    for env_id, steps in ENVS_TO_USE.items():
        print(f"\n=== Default reward baseline: {env_id} ===")
        default_results[env_id] = run_experiment(
            env_id=env_id,
            total_timesteps=steps,
            reward_fn=None,
            label="default",
            seeds=SEEDS,
        )

        print(f"\n=== Human-shaped baseline: {env_id} ===")
        h_fn, h_code = get_human_reward_function(env_id)
        human_codes[env_id] = h_code
        human_results[env_id] = run_experiment(
            env_id=env_id,
            total_timesteps=steps,
            reward_fn=h_fn,
            label="human",
            seeds=SEEDS,
        )

    with open(default_path, "wb") as f:
        pickle.dump(default_results, f)
    with open(human_path, "wb") as f:
        pickle.dump(human_results, f)
    with open(human_codes_path, "wb") as f:
        pickle.dump(human_codes, f)

    print("Saved default + human results.")



=== Default reward baseline: CartPole-v1 ===
[default] env=CartPole-v1 seed=0 steps=100000


KeyboardInterrupt: 

In [ ]:

llm_results_path = out_path("llm_results.pkl")
llm_codes_path = out_path("llm_codes.pkl")
llm_meta_path = out_path("llm_meta.pkl")

if os.path.exists(llm_results_path):
    with open(llm_results_path, "rb") as f:
        llm_results = pickle.load(f)
else:
    llm_results = {}

if os.path.exists(llm_codes_path):
    with open(llm_codes_path, "rb") as f:
        llm_codes = pickle.load(f)
else:
    llm_codes = {}

if os.path.exists(llm_meta_path):
    with open(llm_meta_path, "rb") as f:
        llm_meta = pickle.load(f)
else:
    llm_meta = {}

for env_id, steps in ENVS_TO_USE.items():
    if env_id in llm_results and isinstance(llm_results[env_id], list) and len(llm_results[env_id]) == len(SEEDS):
        print(f"Skipping {env_id}: already has {len(SEEDS)} seeds.")
        continue

    print(f"\n=== LLM reward: {env_id} ===")
    fn, code, meta = get_llm_reward_function(env_id, PROMPTS[env_id])
    llm_codes[env_id] = code
    llm_meta[env_id] = meta

    if meta["status"] != "ok":
        print(f"LLM generation failed for {env_id}: {meta}")
        llm_results[env_id] = []
    else:
        llm_results[env_id] = run_experiment(
            env_id=env_id,
            total_timesteps=steps,
            reward_fn=fn,
            label="llm",
            seeds=SEEDS,
        )

    with open(llm_results_path, "wb") as f:
        pickle.dump(llm_results, f)
    with open(llm_codes_path, "wb") as f:
        pickle.dump(llm_codes, f)
    with open(llm_meta_path, "wb") as f:
        pickle.dump(llm_meta, f)
    print(f"Checkpoint saved after {env_id}.")

print("Saved/updated LLM results + metadata.")


In [ ]:
def final_eval_mean(run, last_n=3):
    vals = run.get("eval_returns", [])
    if len(vals) == 0:
        return np.nan
    return float(np.mean(vals[-last_n:]))


def timesteps_to_threshold(run, threshold):
    for step, ret in zip(run.get("eval_steps", []), run.get("eval_returns", [])):
        if ret >= threshold:
            return int(step)
    return np.nan


def summarize_condition(runs, threshold):
    final_vals = [final_eval_mean(r) for r in runs]
    final_vals = [v for v in final_vals if not np.isnan(v)]

    ttt = [timesteps_to_threshold(r, threshold) for r in runs]
    ttt_valid = [v for v in ttt if not np.isnan(v)]

    success = [1 if not np.isnan(v) else 0 for v in ttt]

    return {
        "n_seeds": len(runs),
        "final_eval_mean": float(np.mean(final_vals)) if final_vals else np.nan,
        "final_eval_std": float(np.std(final_vals)) if final_vals else np.nan,
        "sample_efficiency_median_steps": float(np.median(ttt_valid)) if ttt_valid else np.nan,
        "success_rate": float(np.mean(success)) if success else np.nan,
    }


rows = []
for env_id in ENVS_TO_USE:
    threshold = SUCCESS_THRESHOLDS[env_id]
    env_rows = {
        "default": summarize_condition(default_results.get(env_id, []), threshold),
        "human": summarize_condition(human_results.get(env_id, []), threshold),
        "llm": summarize_condition(llm_results.get(env_id, []), threshold),
    }
    for condition, stats in env_rows.items():
        rows.append({
            "environment": env_id,
            "condition": condition,
            "threshold": threshold,
            **stats,
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(out_path("summary_metrics.csv"), index=False)
print(f"Saved: {out_path('summary_metrics.csv')}")


In [ ]:
EXPECTED_TOKENS = {
    "CartPole-v1": ["obs[2]", "obs[0]"],
    "LunarLander-v3": ["obs[4]", "obs[0]", "obs[3]"],
    "MountainCar-v0": ["obs[0]", "obs[1]"],
}


def classify_llm_failure(env_id, meta, code, runs):
    status = meta.get("status", "unknown")
    if status == "truncation_incomplete":
        return "truncation_incomplete"
    if status != "ok":
        return "syntax_runtime_invalid"

    if any(tok not in code for tok in EXPECTED_TOKENS.get(env_id, [])):
        return "underspecification"

    if len(runs) == 0:
        return "syntax_runtime_invalid"

    train_vals = []
    eval_vals = []
    for r in runs:
        tr = r.get("train_rewards", [])
        ev = r.get("eval_returns", [])
        if len(tr) > 0:
            train_vals.append(float(np.mean(tr[-10:])))
        if len(ev) > 0:
            eval_vals.append(float(np.mean(ev[-3:])))

    train_mean = float(np.mean(train_vals)) if train_vals else np.nan
    eval_mean = float(np.mean(eval_vals)) if eval_vals else np.nan
    threshold = SUCCESS_THRESHOLDS[env_id]

    if not np.isnan(train_mean) and not np.isnan(eval_mean):
        if abs(train_mean) > 5.0 * max(1.0, abs(eval_mean)):
            return "magnitude_mismatch"
        if eval_mean < threshold and train_mean > 0:
            return "reward_hacking"

    return "correct"


failure_rows = []
for env_id in ENVS_TO_USE:
    meta = llm_meta.get(env_id, {})
    code = llm_codes.get(env_id, "")
    runs = llm_results.get(env_id, [])
    label = classify_llm_failure(env_id, meta, code, runs)
    failure_rows.append({
        "environment": env_id,
        "auto_failure_label": label,
        "generation_status": meta.get("status", "missing"),
        "finish_reason": meta.get("finish_reason", "missing"),
        "detail": meta.get("detail", ""),
    })

failure_df = pd.DataFrame(failure_rows)
print(failure_df.to_string(index=False))
failure_df.to_csv(out_path("failure_analysis.csv"), index=False)
print(f"Saved: {out_path('failure_analysis.csv')}")


In [ ]:

CARTPOLE_PROMPT_VARIANTS = {
    "vague": '''
Write a reward function for CartPole-v1.
Use: def custom_reward(obs, action, default_reward, next_obs, done, info):
Return a float.
''',
    "detailed": PROMPTS["CartPole-v1"],
    "hint_based": '''
Write a reward function for CartPole-v1.
Keep the pole upright and keep the cart near center.
Use this exact signature:
def custom_reward(obs, action, default_reward, next_obs, done, info):
Return a float. No imports. No docstrings.
''',
}

variation_path = out_path("variation_results.pkl")
if os.path.exists(variation_path):
    with open(variation_path, "rb") as f:
        variation_results = pickle.load(f)
else:
    variation_results = {}

for name, prompt in CARTPOLE_PROMPT_VARIANTS.items():
    already_done = (
        name in variation_results
        and isinstance(variation_results[name], dict)
        and len(variation_results[name].get("runs", [])) == len(SEEDS)
    )
    if already_done:
        print(f"Skipping prompt variant {name}: already complete.")
        continue

    print(f"\n=== Prompt variant: {name} ===")
    fn, code, meta = get_llm_reward_function("CartPole-v1", prompt)

    data = {
        "meta": meta,
        "code": code,
        "runs": [],
    }
    if meta.get("status") == "ok":
        data["runs"] = run_experiment(
            env_id="CartPole-v1",
            total_timesteps=ENVS_TO_USE["CartPole-v1"],
            reward_fn=fn,
            label=f"llm_prompt_{name}",
            seeds=SEEDS,
        )
    else:
        print(f"Skipped {name}: {meta}")

    variation_results[name] = data
    with open(variation_path, "wb") as f:
        pickle.dump(variation_results, f)
    print(f"Checkpoint saved after prompt {name}.")

var_rows = []
for name, data in variation_results.items():
    stats = summarize_condition(data["runs"], SUCCESS_THRESHOLDS["CartPole-v1"])
    var_rows.append({
        "prompt": name,
        "status": data["meta"].get("status", "missing"),
        **stats,
    })

variation_df = pd.DataFrame(var_rows)
print(variation_df.to_string(index=False))
variation_df.to_csv(out_path("prompt_variation_summary.csv"), index=False)
print(f"Saved: {out_path('prompt_variation_summary.csv')}")


In [ ]:
def aggregate_eval_curve(runs):
    if len(runs) == 0:
        return [], np.array([]), np.array([])
    valid = [r for r in runs if len(r.get("eval_returns", [])) > 0]
    if len(valid) == 0:
        return [], np.array([]), np.array([])

    min_len = min(len(r["eval_returns"]) for r in valid)
    steps = valid[0]["eval_steps"][:min_len]
    arr = np.array([r["eval_returns"][:min_len] for r in valid], dtype=float)
    return steps, arr.mean(axis=0), arr.std(axis=0)


for env_id in ENVS_TO_USE:
    plt.figure(figsize=(10, 5))

    for label, runs, color in [
        ("default", default_results.get(env_id, []), "#1f77b4"),
        ("human", human_results.get(env_id, []), "#2ca02c"),
        ("llm", llm_results.get(env_id, []), "#d62728"),
    ]:
        steps, mean, std = aggregate_eval_curve(runs)
        if len(steps) == 0:
            continue
        plt.plot(steps, mean, label=label, color=color, linewidth=2)
        plt.fill_between(steps, mean - std, mean + std, color=color, alpha=0.2)

    plt.axhline(SUCCESS_THRESHOLDS[env_id], color="black", linestyle="--", linewidth=1, label="success threshold")
    plt.title(f"Default-reward evaluation curves: {env_id}")
    plt.xlabel("Timesteps")
    plt.ylabel("Eval return (default reward)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    out_file = out_path(f"eval_curve_{env_id.replace('-', '_')}.png")
    plt.savefig(out_file, dpi=150)
    plt.show()
    print(f"Saved: {out_file}")


plt.figure(figsize=(10, 5))
for name, color in [("vague", "#ff7f0e"), ("detailed", "#d62728"), ("hint_based", "#9467bd")]:
    runs = variation_results.get(name, {}).get("runs", [])
    steps, mean, std = aggregate_eval_curve(runs)
    if len(steps) == 0:
        continue
    plt.plot(steps, mean, label=name, color=color, linewidth=2)
    plt.fill_between(steps, mean - std, mean + std, color=color, alpha=0.2)

base_steps, base_mean, base_std = aggregate_eval_curve(default_results.get("CartPole-v1", []))
if len(base_steps) > 0:
    plt.plot(base_steps, base_mean, label="default baseline", color="#1f77b4", linewidth=2, linestyle="--")

plt.axhline(SUCCESS_THRESHOLDS["CartPole-v1"], color="black", linestyle="--", linewidth=1)
plt.title("Prompt variation on CartPole-v1 (default-reward eval)")
plt.xlabel("Timesteps")
plt.ylabel("Eval return (default reward)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_path("prompt_variation_eval_curve.png"), dpi=150)
plt.show()
print(f"Saved: {out_path('prompt_variation_eval_curve.png')}")


In [ ]:
artifacts = [
    "default_results.pkl",
    "human_results.pkl",
    "human_codes.pkl",
    "llm_results.pkl",
    "llm_codes.pkl",
    "llm_meta.pkl",
    "summary_metrics.csv",
    "failure_analysis.csv",
    "variation_results.pkl",
    "prompt_variation_summary.csv",
    "prompt_variation_eval_curve.png",
]

print("Expected artifacts in current folder:")
for name in artifacts:
    p = out_path(name)
    print("-", p, "OK" if os.path.exists(p) else "MISSING")
